In [1]:
import pandas as pd
import random

macbeth_df = pd.read_fwf('macbeth.txt')
macbeth_df.columns = ['text']

hamlet_df = pd.read_fwf('hamlet.txt')
hamlet_df.columns = ['text']

# Combine the dataframes
df = pd.concat([macbeth_df, hamlet_df])

In [2]:
df

,text
0,THE TRAGEDY OF MACBETH
1,by William Shakespeare
2,Dramatis Personae
3,"DUNCAN, King of Scotland"
4,"MACBETH, Thane of Glamis and Cawdor, a general..."
...,...
4157,Becomes the field but here shows much amiss.
4158,"Go, bid the soldiers shoot."
4159,Exeunt marching; after the which a peal of ord...
4160,are shot off.


In [3]:
import numpy as np

splits = np.array_split(df, 5)

# Split 0 -> Caesar Cipher

In [4]:
def caesar_encrypt(text, s=7):
    result = ""
    # transverse the plain text
    for i in range(len(text)):
        char = text[i]
        # Encrypt uppercase characters in plain text
        
        if (char.isupper()):
            result += chr((ord(char) + s-65) % 26 + 65)
        # Encrypt lowercase characters in plain text
        else:
            result += chr((ord(char) + s - 97) % 26 + 97)
    return result

In [5]:
caesar_df = pd.DataFrame(columns=['algorithm', 'encrypt'])
caesar_df['encrypt'] = [caesar_encrypt(text, random.randint(0, 26)) for text in splits[0].text.values] 
caesar_df['algorithm'] = 'caesar'

In [6]:
caesar_df

,algorithm,encrypt
0,caesar,QEBkQOXDBAVkLCkJXZYBQE
1,caesar,yvkTfiifxjkPexhbpmbxob
2,caesar,DramatisnPersonae
3,caesar,QHAPNAmaXvatabsaFpbgynaq
4,caesar,RFHGJYMesYmfsjstksLqfrnxsfsisHfbitwesfsljsjwfq...
...,...,...
1361,caesar,HVXWZOCwipVndyzwriOcziixjhznihtiadoivbvdiwiDic...
1362,caesar,apcqpnek
1363,caesar,Ufmjclyqlrfclkypzjcxldmslbcblyqlrfclpmaix
1364,caesar,Nfaoebnqanaqatrarenyanfagurapnfvatanveb


# Split 1 -> Affine Cipher

## Code for Affine Cipher

In [7]:
class Affine(object):
    def __init__(self):
        pass

    def egcd(self, a, b): 
        x,y, u,v = 0,1, 1,0
        while a != 0: 
            q, r = b//a, b%a 
            m, n = x-u*q, y-v*q 
            b,a, x,y, u,v = a,r, u,v, m,n 
        gcd = b 
        return gcd, x, y 

    def modinv(self, a, m): 
        gcd, x, y = self.egcd(a, m) 
        if gcd != 1: 
            return None # modular inverse does not exist 
        else: 
            return x % m 
    
    def encrypt(self, text, key): 
        #E = (a*x + b) % 26 
        return ''.join([ chr(((key[0]*(ord(t) - ord('A')) + key[1] ) % 26) + ord('A')) for t in text.upper().replace(' ', '') ]) 


    def decrypt(self, cipher, key): 
        #D(E) = (a^-1 * (E - b)) % 26
        return ''.join([ chr(((self.modinv(key[0], 26)*(ord(c) - ord('A') - key[1])) % 26) + ord('A')) for c in cipher ]) 
      
affine = Affine()

In [8]:
affine_df = pd.DataFrame(columns=['algorithm', 'encrypt'])
affine_df['encrypt'] = [affine.encrypt(text, (random.randint(0, 26), random.randint(0, 26))) for text in splits[1].text.values]
affine_df['algorithm'] = 'affine'

In [9]:
affine_df

,algorithm,encrypt
0,affine,SNRZTBXCNTASRZMCEDZQRFATSAZMPTNZRRZEDX
1,affine,FTRBTRRRRTXNJFNBPPZNPRZTJXJRLXXZLBHTTRPLZRJJ
2,affine,VDTLTVJPTFTJJPZLJRTPBLJBHPLDBLJPRB
3,affine,TBQHQKYTKVQKTBTSXKTODQB
4,affine,NNNNNNNNNNNNNNNNNNNNNN
...,...,...
1361,affine,UCCKEKAECWCCAAUQKEAQK
1362,affine,NLDXXJDLZVJBLPBLZVBJXZJXDBLLZLBPB
1363,affine,YYMOSGGYUAGIAUIGSIEAYKIWUGMUEWSC
1364,affine,YYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYY


# Split 2 -> Vigenere Cipher

In [10]:
MIN_KEY_LEN = 3
MAX_KEY_LEN = 10
ALPHA = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

class Vigenere:
    def __init__(self, key=None):
        if key is None:
            key = self.generate_random_key()
        self.key = key

    def get_key(self):
        return self.key
    
    def set_key(self):
        key = self.generate_random_key()

    def generate_random_key(self) -> str:
        length = random.randint(MIN_KEY_LEN, MAX_KEY_LEN + 1)
        return ''.join(random.sample(ALPHA, length))

    def generate_keyword(self, n: int) -> str:
        return ((n // len(self.key) + 1) * self.key)[:n]

    def get_ord(self, letter):
        return ALPHA.index(letter.upper())
    
    def get_chr(self, num, uppercase=True):
        return ALPHA[num % 26] if uppercase else ALPHA[num % 26].lower()
    
    def encrypt(self, plaintext: str) -> str:
        keyword = self.generate_keyword(len(plaintext))
        return ''.join([self.get_chr(self.get_ord(ch) + self.get_ord(keyword[i]), ch.isupper())
                        if ch.upper() in ALPHA else ch for i, ch in enumerate(plaintext)])

    def decrypt(self, ciphertext: str) -> str:
        keyword = self.generate_keyword(len(ciphertext))
        return ''.join([self.get_chr(self.get_ord(ch) - self.get_ord(keyword[i]), ch.isupper())
                        if ch.upper() in ALPHA else ch for i, ch in enumerate(ciphertext)])

In [11]:
vigenere_test = Vigenere('TESTING')
plaintext = 'this is a test. Hello!'
ciphertext = vigenere_test.encrypt(plaintext)
print(plaintext)
print(ciphertext)

vigenere_test.decrypt(ciphertext)

this is a test. Hello!
mlal vy e mmfz. Zxtyu!


'this is a test. Hello!'

In [18]:
vigenere = Vigenere()

vigenere_df = pd.DataFrame(columns=['algorithm', 'encrypt'])

def vigenere_encrypt_with_random_key(vigenere, text):
    vigenere.set_key()
    return vigenere.encrypt(text)

vigenere_df['encrypt'] = [vigenere_encrypt_with_random_key(vigenere, text) for text in splits[2].text.values]
vigenere_df['algorithm'] = 'vigenere'

vigenere_df

,algorithm,encrypt
0,vigenere,"Tete up tl wxpce tea jenrpep oc tees jiddt,"
1,vigenere,"Qdaq, fb xcafj qdip amlaoetfkn yoja,"
2,vigenere,Ea jwy wpmnosa lqr aybo xjd opbwk po et.
3,vigenere,"Ekr. Troh, troh, 'qsiih kkt wpmaao."
4,vigenere,"Yar. Sfp akwk atdiia,"
...,...,...
1360,vigenere,Ewm. Tewt uor mrot pexyh ie. Brp iat ie yokfuo...
1361,vigenere,"lb lqr beihotohfl, xy phb cljsljakyy kf kuo yl..."
1362,vigenere,"lxlfcaqeok oc orn breo-poasbnvbz ikvb, xjd xy ..."
1363,vigenere,"yatqar lrllopar yorhd yhxngb ylq tetewl, bb es..."


# Split 4 -> Railfence Cipher

In [13]:
class RailFence(object):
    def __init__(self):
        pass

    def matrix(self, text, key): 
        try:
            rail = [['\n' for i in range(len(text))] for j in range(key)]
            dir_down = False
            row, col = 0, 0

            for i in range(len(text)):
                if (row == 0) or (row == key - 1):
                    dir_down = not dir_down
                rail[row][col] = text[i]
                col += 1
                if dir_down:
                    row += 1
                else:
                    row -= 1
        except:
            print("ERROR", text, key)

        return rail
    
    def encrypt(self, text, key): 

        rail=self.matrix(text,key)
        result = []
        for i in range(key):
            for j in range(len(text)):
                if rail[i][j] != '\n':
                    result.append(rail[i][j])
        return("" . join(result))

        
rail_fence = RailFence()

In [14]:
rail_fence_df = pd.DataFrame(columns=['algorithm', 'encrypt'])
rail_fence_df['encrypt'] = [rail_fence.encrypt(text, random.randint(2, len(text))) for text in splits[4].text.values]
rail_fence_df['algorithm'] = 'rail_fence'


In [15]:
rail_fence_df

,algorithm,encrypt
0,rail_fence,K its?WnAurh?g peee.pr
1,rail_fence,Heacrm t.Aa iN.nonte twahee rsei heeh eeart...
2,rail_fence,convocation of politic worms are e'en at rhuio...
3,rail_fence,only emperor for diet. We fat all creaturdensa...
4,rail_fence,wYge oeg .ubafsr rat ntofa gaeogtlua rmkrs iu...
...,...,...
1360,rail_fence,Breeech o smtheuo.sbws sstd ihlmmeeua ic fh
1361,rail_fence,"Go, bid t.hteo oshosl dsire"
1362,rail_fence,E nx;ahaacegfc neuntipdnieherthrwao c l mrte...
1363,rail_fence,are s.hfofto


# Exporting to CSV

In [16]:
def export_df(df, filename):
    df.to_csv(filename, index=False)

# combine all the dataframes
combined_df = pd.concat([caesar_df, affine_df, vigenere_df, rail_fence_df])

# export the combined dataframe
export_df(combined_df, 'dataset.csv')